# Ionization region model

The ionization region model (IRM) describes the dense plasma of a HiPIMS pulse in the
ionization region (IR) over the racetrack of a sputtering magnetron: an annulus between
the radii $r_1$ and $r_2$, from the top of the cathode sheath at $z_1$ to $z_2$ above the
target. It is volume averaged and time dependent, and it is driven by the measured
discharge current $I_D(t)$ and voltage $V_D(t)$ (Raadu et al. 2011, Huo et al. 2017,
Gudmundsson et al. 2022 for copper). `carafe.irm.IonizationRegion` follows their structure.

This tutorial builds a copper region, runs a short pulse, looks at what the IRM papers
report (the current carried by every species, $\alpha_t$, $\beta_t$, the ionized flux
fraction, the rarefaction of the gas, the power balance of the electrons) and fits the two
parameters of the model to a discharge current. The rate coefficients, yields and
the geometry are **placeholders** of the right order of magnitude, not data: the
procedure is the IRM's, the numbers are not.

In [ ]:
import warnings

import numpy as np
import plasma_plots

from carafe.irm import IonizationRegion, pulse_averages, waveform

plasma_plots.set_backend("plotly")
plasma_plots.set_math("latex")
warnings.filterwarnings("ignore", message=".*placeholders, not data")

## Species and groups

Two electron groups: the thermal electrons `e`, a Maxwellian with an energy equation for
$T_e$, and the hot electrons `e_h`, the secondary electrons from the target accelerated in
the cathode sheath, at a fixed temperature $T_H$; a hot electron becomes a thermal one when
it has lost $\tfrac32 T_H$ in collisions. Argon comes in three ground-state groups: the cold
filling gas `Ar`, the warm `Ar_w` (ions that left for the bulk plasma and come back
neutralized) and the hot `Ar_h` (ions neutralized and reflected at the target, and gas
kicked by the sputtered atoms), with the metastables `Ar_m`, `Ar+` and `Ar2+`. The metal is
`Cu`, `Cu+` and `Cu2+`. Quasi-neutrality gives the thermal electrons,
$n_e = n_{\mathrm{Ar}^+} + 2 n_{\mathrm{Ar}^{2+}} + n_{\mathrm{Cu}^+} + 2 n_{\mathrm{Cu}^{2+}} - n_h$.

In [ ]:
region = IonizationRegion(
    inner_radius="6 mm",
    outer_radius="20 mm",
    sheath_height="2 mm",
    height="28 mm",
    pressure="0.5 Pa",
    back_attraction=0.5,
    voltage_fraction=0.15,
    pulse_end="40 us",
)
list(region.species)

The volume reactions are ionization, excitation (into the metastables and into radiating
levels) and elastic collisions of the electrons, quenching of the metastables, charge
exchange $\mathrm{Ar^+ + Cu \to Ar + Cu^+}$, Penning ionization of copper by the metastables
and the cold gas kicked out by the sputtered atoms:

In [ ]:
[reaction.name for reaction in region.reactions()]

The geometry: the volume, the racetrack area $S_\mathrm{RT}$ under the region and the area
$S_\mathrm{BP}$ towards the bulk plasma (the top and the two sides), and the filling density.

In [ ]:
region.volume, region.target_area, region.bulk_area, region.gas_density

## Back-attraction and ohmic heating

Ions leave the region at the Bohm speed through $S_\mathrm{BP}$. Most of the discharge
voltage falls over the cathode sheath, but a share $f$ of it, $V_\mathrm{IR} = f V_D$,
falls over the region itself and pulls a share $\beta_t$ of the ions that leave back to
the target, so that the flux to the target is $\beta_t / (1 - \beta_t)$ times the flux to
the bulk. At the target an ion of charge $Z$ arrives with $Z e (1 - f) V_D$, sputters
$Y = a E^b$ copper atoms and emits secondary electrons, which become hot electrons. The
thermal electrons are heated ohmically by the current through the region,
$P = I_D f V_D$. After the pulse (`pulse_end`) no ion is attracted back, $\beta_t = 0$.

$\beta_t$ and $f$ are the two fitting parameters of the IRM: the model computes its own
discharge current at the target, $I_D^\mathrm{model} = e (\sum_i Z_i \Gamma_i^t +
\Gamma_\mathrm{SE})$, and the parameters are chosen so that it matches the measured one.

## A copper pulse

A 40 µs pulse that rises to 20 A at 500 V, followed by 40 µs of afterglow. The region is
*current driven*: the given $I_D(t)$ heats the electrons.

In [ ]:
on = 40.0e-6


def current(t):
    return 20.0 * (1.0 - np.exp(-t / 12.0e-6)) if t <= on else 0.0


def voltage(t):
    return 500.0 if t <= on else 0.0


times = np.linspace(0.0, 2.0 * on, 161)
data = region.evolve(times, current, voltage, region.initial_state(1.0e17))
data = data.isel(t=slice(1, None))  # the seed state at t = 0 is not a discharge
data

The densities, the electron temperature, the current at the target by carrier and the
outputs of the IRM over the time. $\alpha_t$ is the share of the sputtered atoms that are
ionized and return to the target, the ionized flux fraction the share of ions in the copper
that leaves for the bulk.

In [ ]:
species = ["e", "Ar", "Ar_h", "Ar_w", "Ar_m", "Ar+", "Cu", "Cu+", "Cu2+"]
with plasma_plots.figure(4, 1, figsize=(7, 14)) as fig:
    data.density.sel(species=species).plasma.plot.profiles(
        x="t", over="species", logy=True, ax=fig[0]
    )
    data.Te.plasma.plot.timeseries(logy=False, title="electron temperature", ax=fig[1])
    data.target_current.plasma.plot.profiles(
        x="t", over="carrier", logy=False, ax=fig[2]
    )
    data.alpha_t.plasma.plot.timeseries(
        data.back_attraction,
        data.ionized_flux_fraction,
        data.rarefaction,
        logy=False,
        title="IRM outputs",
        ax=fig[3],
    )

The current at the target is carried by every ion species and the secondary electrons,
and the sum is the model's current (its shares near the end of the pulse below). With these
placeholder rates the copper ions carry most of it: the discharge is dominated by
self-sputtering.

In [ ]:
peak = data.sel(t=0.95 * on, method="nearest")
shares = peak.target_current / peak.model_current
{
    str(c): round(float(s), 3)
    for c, s in zip(shares.carrier.values, shares.values, strict=True)
}

## The power balance of the electrons

The ohmic heating of the thermal electrons goes into the collisions (ionization,
excitation, elastic), into the electrons that leave for the bulk, $2 T_e$ each, and into
the energy stored, $\frac{d}{dt}(\tfrac32 n_e T_e e V)$:
$P_\mathrm{ohmic} = P_\mathrm{coll} + P_\mathrm{escape} + P_\mathrm{stored}$. The hot
electrons add the power they lose in collisions, `hot_power`.

In [ ]:
with plasma_plots.figure(1, 1, figsize=(7, 4)) as fig:
    data.electron_power.plasma.plot.profiles(x="t", over="term", logy=False, ax=fig[0])

In [ ]:
pulse = data.sel(t=slice(5.0e-6, on))
ohmic = float(np.trapezoid(pulse.electron_power.sel(term="ohmic"), pulse.t))
hot = float(np.trapezoid(pulse.hot_power, pulse.t))
ohmic, hot  # J deposited in the pulse by the thermal and the hot electrons

## Time averages

The IRM papers give the time averages over the pulse and the afterglow, counting atoms:
$\langle\alpha_t\rangle$ (ionized and back-attracted over sputtered),
$\langle\beta_t\rangle$ (back-attracted over all metal ions that leave, below $\beta_t$ of
the pulse since $\beta_t = 0$ in the afterglow), $\langle\alpha\rangle$ with
$\langle\alpha_t\rangle = \langle\alpha\rangle\langle\beta_t\rangle$, the ionized flux
fraction and the deepest rarefaction of the gas.

In [ ]:
{key: round(value, 3) for key, value in pulse_averages(data).items()}

## Voltage-driven or current-driven

With `current=None` the model's own current heats the electrons: a discharge driven by the
voltage alone, whose current is an output. With self-sputtering a long pulse at a fixed
voltage can run away, as a real one can; a current-driven run cannot.

In [ ]:
driven = region.evolve(times, None, voltage, region.initial_state(1.0e17))
driven = driven.isel(t=slice(1, None))
with plasma_plots.figure(1, 1, figsize=(7, 4)) as fig:
    driven.model_current.plasma.plot.timeseries(
        logy=False,
        reference={"current-driven": data.model_current},
        title="discharge current at 500 V",
        ax=fig[0],
    )

## Fitting $\beta_t$ and $f$

`IonizationRegion.fit` runs the model on the measured waveforms for each trial
$(\beta_t, f)$ and minimizes the mismatch of the currents by least squares. To check it,
take a synthetic "measurement": a voltage-driven run of a region with $\beta_t = 0.5$ and
$f = 0.18$, whose current that region reproduces when driven by it (up to the linear
interpolation between the samples). A fit started at $(0.4, 0.14)$ recovers the two
parameters.

In [ ]:
short = 10.0e-6
true = IonizationRegion(back_attraction=0.5, voltage_fraction=0.18, pulse_end=short)
samples = np.linspace(0.0, 2.0 * short, 41)
volts = np.where(samples <= short, 400.0, 0.0)
seed = true.initial_state(1.0e18)
synthetic = true.evolve(samples, None, waveform(samples, volts), seed)

fit = IonizationRegion(pulse_end=short).fit(
    samples,
    synthetic.model_current.values,
    volts,
    seed,
    guess=(0.4, 0.14),
    tolerance=1.0e-4,
)
fit.back_attraction, fit.voltage_fraction, fit.relative_rms

In [ ]:
with plasma_plots.figure(1, 1, figsize=(7, 4)) as fig:
    fit.data.model_current.plasma.plot.timeseries(
        logy=False,
        reference={"synthetic measurement": synthetic.model_current},
        title="fitted discharge current",
        ax=fig[0],
    )

## A measured case

To fit a measured pulse, put its waveforms, the device and the published results in one
CSV file and read it with `carafe.irm.load_case`: `examples/data/irm_copper_case1.csv`
holds case I of Gudmundsson et al. (2022), with placeholder waveforms until the digitized
ones of their Fig. 2 replace them, and the full example
[Ionization region model](/carafe/full-examples/irm-copper-pulse/) fits it and compares
the time averages with their Table 5.